In [5]:
!pip install -q gradio transformers peft bitsandbytes  accelerate

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, PreTrainedTokenizerFast
from peft import PeftModel
import json
import os
import traceback

# ==========================================
# 0. CẤU HÌNH ĐƯỜNG DẪN CHÍNH XÁC (KAGGLE)
# ==========================================
TOKENIZER_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/bpe_vi_tokenizer"
DATASET_PATH = "/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl"

GRU_WEIGHTS_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/gru_nwp_v2_epoch_5.pt"
TCN_WEIGHTS_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/tcn_vn_model_final.pth"
# ĐÃ CẬP NHẬT ĐƯỜNG DẪN MỚI CHO TCN VOCAB
TCN_VOCAB_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/vocab_train.json"  
QWEN_LORA_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/qwen_lora_adapter"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. LOAD TOKENIZER & VOCABULARY (Biến toàn cục)
# ==========================================
print("Đang tải Tokenizer BPE...")
fast_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_PATH)
pad_idx = fast_tokenizer.pad_token_id

class Vocabulary:
    def __init__(self):
        self.pad_token, self.pad_idx = '<pad>', 0
        self.unk_token, self.unk_idx = '<unk>', 1
        self.sos_token, self.sos_idx = '<sos>', 2
        self.eos_token, self.eos_idx = '<eos>', 3
        self.word2idx = {self.pad_token: 0, self.unk_token: 1, self.sos_token: 2, self.eos_token: 3}
        self.idx2word = {0: self.pad_token, 1: self.unk_token, 2: self.sos_token, 3: self.eos_token}

    def load_from_dict(self, word2idx):
        self.word2idx = word2idx
        self.idx2word = {int(v): k for k, v in word2idx.items()}

    def encode(self, text, add_sos_eos=True):
        tokens = text.split()
        seq = [self.word2idx.get(w, self.unk_idx) for w in tokens]
        if add_sos_eos:
            seq = [self.sos_idx] + seq + [self.eos_idx]
        return seq

    def decode(self, indices):
        return " ".join([self.idx2word.get(idx, self.unk_token) for idx in indices])

tcn_vocab = Vocabulary()
if os.path.exists(TCN_VOCAB_PATH):
    with open(TCN_VOCAB_PATH, 'r', encoding='utf-8') as f:
        raw_vocab_data = json.load(f)
        if isinstance(raw_vocab_data, dict) and 'word2idx' in raw_vocab_data:
            word2idx_dict = raw_vocab_data['word2idx']
        else:
            word2idx_dict = raw_vocab_data
            
        tcn_vocab.load_from_dict(word2idx_dict)
        
    print(f"✅ Đã nạp từ điển TCN. Kích thước: {len(tcn_vocab.word2idx)} từ.")
else:
    print(f"❌ KHÔNG TÌM THẤY TỪ ĐIỂN TCN TẠI: {TCN_VOCAB_PATH}")

# ==========================================
# 2. KIẾN TRÚC MÔ HÌNH
# ==========================================
class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.fc.weight = self.embedding.weight

    def forward(self, input_ids, hidden=None):
        embedded = self.dropout(self.embedding(input_ids))
        output, hidden = self.gru(embedded, hidden)
        return self.fc(self.dropout(output)), hidden

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.act1, self.drop1, self.conv2, self.chomp2, self.act2, self.drop2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.GELU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            padding = (kernel_size - 1) * dilation_size
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, 1, dilation_size, padding, dropout))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

class TCNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.tcn = TemporalConvNet(embed_size, num_channels, kernel_size, dropout)
        self.decoder = nn.Linear(num_channels[-1], vocab_size)
        if num_channels[-1] == embed_size:
            self.decoder.weight = self.embedding.weight

    def forward(self, x):
        emb = self.embedding(x)
        y = self.tcn(emb.transpose(1, 2)).transpose(1, 2)
        return self.decoder(y)

# ==========================================
# 3. KHỞI TẠO MÔ HÌNH GRU VÀ TCN
# ==========================================
gru_model = GRULanguageModel(30000, 1024, 1024, 2, 0.2, pad_idx).to(device)
if os.path.exists(GRU_WEIGHTS_PATH):
    checkpoint = torch.load(GRU_WEIGHTS_PATH, map_location=device)
    if 'model_state_dict' in checkpoint:
        gru_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        gru_model.load_state_dict(checkpoint)
    print("✅ Đã nạp trọng số GRU.")

tcn_vocab_size = len(tcn_vocab.word2idx) if len(tcn_vocab.word2idx) > 4 else 30000
tcn_model = TCNLanguageModel(tcn_vocab_size, 256, [128, 256, 512, 256], 3).to(device)
if os.path.exists(TCN_WEIGHTS_PATH):
    tcn_model.load_state_dict(torch.load(TCN_WEIGHTS_PATH, map_location=device))
    print("✅ Đã nạp trọng số TCN.")

# ==========================================
# 4. HÀM DỰ ĐOÁN (CÓ PYVI VÀ BẮT LỖI)
# ==========================================
def pred_gru(seed_text, beam_width, max_length):
    try:
        gru_model.eval()
        try:
            from pyvi import ViTokenizer
            processed_text = ViTokenizer.tokenize(seed_text)
        except ImportError:
            processed_text = seed_text
            
        input_ids = fast_tokenizer.encode(processed_text, return_tensors='pt').to(device)
        device_type = 'cuda' if 'cuda' in device else 'cpu'
        
        with torch.no_grad():
            with torch.autocast(device_type=device_type):
                logits, hidden = gru_model(input_ids, None)
            log_probs = torch.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)
            top_log_probs, top_indices = torch.topk(log_probs, int(beam_width))
            
        base_seq = input_ids[0].tolist()
        beams = [(top_log_probs[i].item(), base_seq + [top_indices[i].item()], hidden) for i in range(int(beam_width))]
        
        for _ in range(int(max_length) - 1):
            new_beams = []
            for score, seq, curr_hidden in beams:
                if seq[-1] == fast_tokenizer.eos_token_id:
                    new_beams.append((score, seq, curr_hidden))
                    continue
                input_tensor = torch.tensor([[seq[-1]]]).to(device)
                
                with torch.no_grad():
                    with torch.autocast(device_type=device_type):
                        logits, next_hidden = gru_model(input_tensor, curr_hidden)
                    log_probs = torch.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)
                
                top_log_probs, top_indices = torch.topk(log_probs, int(beam_width))
                for i in range(int(beam_width)):
                    new_beams.append((score + top_log_probs[i].item(), seq + [top_indices[i].item()], next_hidden))
                    
            beams = sorted(new_beams, key=lambda x: x[0] / (((5 + len(x[1])) / 6) ** 0.7), reverse=True)[:int(beam_width)]
            if all(seq[-1] == fast_tokenizer.eos_token_id for _, seq, _ in beams):
                break
                
        best_seq = beams[0][1]
        decoded = fast_tokenizer.decode(best_seq, skip_special_tokens=True)
        suggested_part = decoded[len(processed_text):].strip()
        return suggested_part.replace("_", " ")

    except Exception as e:
        return f"❌ LỖI GRU:\n{str(e)}\n\nTraceback:\n{traceback.format_exc()}"


def pred_tcn(seed_text, beam_width, max_length, alpha=0.7):
    try:
        tcn_model.eval()
        try:
            from pyvi import ViTokenizer
            processed_text = ViTokenizer.tokenize(seed_text)
        except ImportError:
            processed_text = seed_text
        
        raw_encoded = tcn_vocab.encode(processed_text, add_sos_eos=False)
        input_ids = [tcn_vocab.sos_idx] + raw_encoded
        beams = [(input_ids, 0.0)]

        for _ in range(int(max_length)):
            new_beams = []
            for seq, score in beams:
                if seq[-1] == tcn_vocab.eos_idx:
                    new_beams.append((seq, score))
                    continue

                context_seq = seq[-32:] 
                input_tensor = torch.tensor([context_seq], dtype=torch.long).to(device)

                with torch.no_grad():
                    device_type = 'cuda' if 'cuda' in device else 'cpu'
                    with torch.autocast(device_type=device_type):
                        logits = tcn_model(input_tensor)
                        last_token_logits = logits[0, -1, :]
                        last_token_logits[seq[-1]] -= 3.0
                        if len(seq) > 1:
                            last_token_logits[seq[-2]] -= 1.0
                        log_probs = F.log_softmax(last_token_logits, dim=-1)

                top_lp, top_idx = torch.topk(log_probs, int(beam_width))
                for i in range(int(beam_width)):
                    new_seq = seq + [top_idx[i].item()]
                    new_beams.append((new_seq, score + top_lp[i].item()))

            beams = sorted(new_beams, key=lambda x: x[1] / (((5 + len(x[0])) / 6) ** alpha), reverse=True)[:int(beam_width)]
            if all(b[0][-1] == tcn_vocab.eos_idx for b in beams): 
                break

        best_seq = beams[0][0]
        filtered_seq = [idx for idx in best_seq if idx not in [tcn_vocab.sos_idx, tcn_vocab.eos_idx, tcn_vocab.pad_idx]]
        decoded = tcn_vocab.decode(filtered_seq)
        suggested_part = decoded[len(processed_text):].strip()
        return suggested_part.replace("_", " ")

    except Exception as e:
        return f"❌ LỖI TCN:\n{str(e)}\n\nTraceback:\n{traceback.format_exc()}"


# ==========================================
# 5. QWEN2 LOGIC
# ==========================================
qwen_model, qwen_tokenizer = None, None

def load_qwen():
    global qwen_model, qwen_tokenizer
    if qwen_model is None:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",               
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True            
        )
        try:
            qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_LORA_PATH)
            if qwen_tokenizer.pad_token is None:
                qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
                
            base = AutoModelForCausalLM.from_pretrained(
                "Qwen/Qwen2-1.5B", 
                quantization_config=bnb_config, 
                device_map="auto",
                trust_remote_code=True
            )
            qwen_model = PeftModel.from_pretrained(base, QWEN_LORA_PATH)
            qwen_model.eval()
            return "✅ Tải mô hình Qwen thành công!"
        except Exception as e: 
            return f"❌ Lỗi: {str(e)}"
    return "✅ Qwen đã sẵn sàng."

def pred_qwen(prompt_text, b, m):
    if qwen_model is None: 
        return "Hãy nhấn tải Qwen trước!"
    try:
        from pyvi import ViTokenizer
        processed_prompt = ViTokenizer.tokenize(prompt_text)
    except ImportError:
        processed_prompt = prompt_text.replace(" ", "_")
    
    inputs = qwen_tokenizer(processed_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outs = qwen_model.generate(
            **inputs, 
            max_new_tokens=int(m), 
            num_beams=int(b), 
            no_repeat_ngram_size=2,
            pad_token_id=qwen_tokenizer.pad_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
            do_sample=False
        )
        
    full_text = qwen_tokenizer.decode(outs[0], skip_special_tokens=True)
    suggested_part = full_text[len(processed_prompt):].strip()
    return suggested_part.replace("_", " ")


# ==========================================
# 6. GIAO DIỆN GRADIO
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Hệ Thống Dự Đoán Từ Tiếp Theo")
    with gr.Tabs():
        with gr.TabItem("1️⃣ GRU"):
            with gr.Row():
                with gr.Column():
                    in_g = gr.Textbox(label="Nhập câu", placeholder="Ví dụ: bộ giáo dục và đào tạo")
                    b_g = gr.Slider(1, 10, value=3, label="Beam Width", step=1)
                    l_g = gr.Slider(5, 50, value=15, label="Max Length", step=1)
                    btn_g = gr.Button("Dự đoán", variant="primary")
                out_g = gr.Textbox(label="Kết quả GRU", lines=6)
            btn_g.click(pred_gru, [in_g, b_g, l_g], out_g)
            
        with gr.TabItem("2️⃣ TCN"):
            with gr.Row():
                with gr.Column():
                    in_t = gr.Textbox(label="Nhập câu", placeholder="Ví dụ: bộ giáo dục và đào tạo")
                    b_t = gr.Slider(1, 10, value=3, label="Beam Width", step=1)
                    l_t = gr.Slider(5, 50, value=15, label="Max Length", step=1)
                    btn_t = gr.Button("Dự đoán", variant="primary")
                out_t = gr.Textbox(label="Kết quả TCN", lines=6)
            btn_t.click(pred_tcn, [in_t, b_t, l_t], out_t)
            
        with gr.TabItem("3️⃣ Qwen2"):
            status = gr.Textbox(label="Trạng thái", value="Chưa tải Qwen")
            gr.Button("Tải Qwen").click(load_qwen, outputs=status)
            with gr.Row():
                with gr.Column():
                    in_q = gr.Textbox(label="Nhập câu")
                    b_q = gr.Slider(1, 10, value=3, label="Beam Width", step=1)
                    l_q = gr.Slider(5, 100, value=30, label="Max Length", step=1)
                    btn_q = gr.Button("Dự đoán", variant="primary")
                out_q = gr.Textbox(label="Kết quả Qwen (Gợi ý tiếp theo)", lines=6)
            btn_q.click(pred_qwen, [in_q, b_q, l_q], out_q)

demo.launch(share=True)

Đang tải Tokenizer BPE...
✅ Đã nạp từ điển TCN. Kích thước: 30000 từ.
✅ Đã nạp trọng số GRU.
✅ Đã nạp trọng số TCN.


/tmp/ipykernel_57/3599099624.py:327: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://21b8a0be33c6cdb9e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
